# OnioRakshak — RAG Knowledge Base: Build & Test Retrieval

This notebook builds the vector index over the curated onion knowledge
base and tests retrieval quality before it's used by the chatbot
(`onion_qa_bot.py`).

**Why RAG instead of just prompting the model:** a plain prompted LLM
answers from its general training data, which can drift, hallucinate
specifics, or simply not reflect this project's own curated sources.
RAG (Retrieval-Augmented Generation) instead retrieves the most relevant
passages from `onion_knowledge_base.json` at question-time and grounds
the model's answer in them — meaning answers can be traced back to a
specific source chunk, and the knowledge base can be corrected or
expanded later without retraining or reworking the prompt.

**Note on file organization:** the actual retrieval logic lives in
`rag_pipeline.py` (a reusable module, since the live chatbot needs to
import and call it repeatedly) — this notebook is for building the index
once and verifying retrieval quality, not for holding the production
logic itself.


In [1]:
import sys
sys.path.append("..")  # so we can import rag_pipeline.py from ml/

from rag_pipeline import OnionKnowledgeBase
import json


## 1. Load and Inspect the Knowledge Base

**Why this matters:** before embedding anything, it's worth confirming
the knowledge base actually has the coverage we think it does — the
retrieval quality can only ever be as good as what's in here.


In [3]:
with open("onion_knowledge_base.json", "r", encoding="utf-8") as f:
    kb_raw = json.load(f)

print(f"{len(kb_raw)} chunks in the knowledge base:\n")
for chunk in kb_raw:
    print(f"- {chunk['id']}: {chunk['topic']}")


10 chunks in the knowledge base:

- storage_conditions: Ideal Storage Conditions
- curing_process: Curing Onions Before Storage
- common_diseases: Common Onion Diseases and Pests
- storage_spoilage_causes: What Causes Onions to Spoil in Storage
- onion_varieties_india: Onion Varieties Grown in India
- market_dynamics: Onion Market Dynamics in India
- nutrition: Onion Nutrition
- why_onions_cause_tears: Why Cutting Onions Causes Tears
- cooking_uses: Onions in Cooking
- oniorakshak_project: About the OnioRakshak Project


## 2. Build the Vector Index

**Why this matters:** each chunk needs to be converted into a numeric
embedding vector before similarity search is possible. This uses
Gemini's `gemini-embedding-001` model with `task_type=RETRIEVAL_DOCUMENT`
(as opposed to `RETRIEVAL_QUERY`, used later for questions) — Gemini's
embedding model treats these differently internally, since a document
chunk and a search query serve different roles even when their content
overlaps semantically.

The index is saved to disk (`onion_kb_index.faiss` +
`onion_kb_metadata.json`) so this embedding step — which costs a small
amount of API quota — doesn't need to re-run every time the chatbot
starts up. Re-run this cell with `force_rebuild=True` only after editing
the knowledge base JSON.


In [4]:
kb = OnionKnowledgeBase()
kb.build_index()  # loads from disk if already built; pass force_rebuild=True to redo
print("Index ready.")


Index ready.


## 3. Test Retrieval Quality

**Why this matters:** a RAG pipeline is only as useful as its retrieval
step — if the wrong chunks get retrieved, the model will confidently
answer from irrelevant context. This section runs a handful of test
queries covering different knowledge base topics, and checks that the
right chunk actually comes back on top.


In [5]:
test_queries = [
    "why are my onions rotting in storage",
    "which onion variety should I grow for long storage",
    "why do I cry when cutting an onion",
    "when are onion prices usually lowest in the year",
    "are onions healthy to eat",
    "what pests damage onion crops",
]

for q in test_queries:
    print(f"Query: {q}")
    results = kb.retrieve(q, top_k=2)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['topic']}")
    print()


Query: why are my onions rotting in storage
  [0.777] What Causes Onions to Spoil in Storage
  [0.757] Curing Onions Before Storage

Query: which onion variety should I grow for long storage
  [0.711] Curing Onions Before Storage
  [0.698] Onion Varieties Grown in India

Query: why do I cry when cutting an onion
  [0.815] Why Cutting Onions Causes Tears
  [0.691] Onion Nutrition

Query: when are onion prices usually lowest in the year
  [0.772] Onion Market Dynamics in India
  [0.693] Onion Varieties Grown in India

Query: are onions healthy to eat
  [0.747] Onion Nutrition
  [0.710] Onions in Cooking

Query: what pests damage onion crops
  [0.764] Common Onion Diseases and Pests
  [0.692] What Causes Onions to Spoil in Storage



**Reading this:** each query's top-scoring result should topically match
what's being asked (e.g. the rotting query should surface
`storage_spoilage_causes`, the pests query should surface
`common_diseases`). If a query surfaces an unrelated chunk as the top
result, that's a sign the knowledge base needs a chunk written more
directly to that phrasing, or the relevance threshold in
`onion_qa_bot.py` needs adjusting.


## 4. Test the Relevance Threshold

**Why this matters:** `onion_qa_bot.py` only injects retrieved chunks
into the prompt when their similarity score clears 0.5 — otherwise it
lets Gemini answer from general knowledge instead of forcing in a weak,
irrelevant match. This section checks that threshold against a query
that's onion-related but NOT covered by any knowledge base chunk, to
confirm weak matches are correctly filtered out rather than forced in.


In [6]:
off_topic_for_kb_query = "what is the botanical family of onions"
results = kb.retrieve(off_topic_for_kb_query, top_k=3)

print(f"Query: {off_topic_for_kb_query}\n")
for r in results:
    flag = "PASSES threshold (>= 0.5)" if r["score"] >= 0.5 else "filtered out (< 0.5)"
    print(f"  [{r['score']:.3f}] {r['topic']} -- {flag}")


Query: what is the botanical family of onions

  [0.692] Onion Nutrition -- PASSES threshold (>= 0.5)
  [0.674] Onions in Cooking -- PASSES threshold (>= 0.5)
  [0.666] Common Onion Diseases and Pests -- PASSES threshold (>= 0.5)


**Reading this:** since nothing in the knowledge base directly covers
onion taxonomy, we'd expect scores here to be lower than the on-topic
queries above — confirming the threshold correctly avoids forcing in a
loosely-related chunk when Gemini's own general knowledge would answer
better.


## 5. Summary

- Knowledge base: 10 curated chunks covering storage, curing, diseases,
  spoilage causes, varieties, market dynamics, nutrition, cooking, and
  the OnioRakshak project itself
- Embeddings via `gemini-embedding-001`, indexed with FAISS
  (`IndexFlatIP` over normalized vectors, i.e. cosine similarity)
- Retrieval verified against on-topic queries (correct chunks surfaced)
  and an off-topic-for-the-KB query (correctly scored lower)
- Next: `05_chatbot_testing.ipynb` tests the full chatbot (rule-based +
  RAG + Gemini generation, multilingual) end-to-end
